In [36]:
# Biblioteker
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects
from scipy.stats import chi2

In [37]:
# Indlæs data
gdp = pd.read_csv("GDP.csv", skiprows=4)
aging = pd.read_csv("age-dependency-ratio-old.csv")
pwt = pd.read_excel("Human-Productivity.xlsx", sheet_name="Data")
rd = pd.read_csv("R&D.csv", skiprows=4)

In [38]:
# Tjek de har læst korrekt - Kan sletts, kun til kontrol
print(gdp.head())
print(gdp.columns[:10])

print(aging.head())
print(aging.columns)

print(pwt.head())
print(pwt.columns[:15])

print(rd.head())
print(rd.columns[:15])

                  Country Name Country Code  \
0                        Aruba          ABW   
1  Africa Eastern and Southern          AFE   
2                  Afghanistan          AFG   
3   Africa Western and Central          AFW   
4                       Angola          AGO   

                                  Indicator Name     Indicator Code  1960  \
0  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   
1  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   
2  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   
3  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   
4  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   

   1961  1962  1963  1964  1965  ...          2017          2018  \
0   NaN   NaN   NaN   NaN   NaN  ...  37524.914920  39287.059713   
1   NaN   NaN   NaN   NaN   NaN  ...   3837.726375   3723.216423   
2   NaN   NaN   NaN   NaN   NaN  ...   2335.795862

In [39]:
# Clean GDP data
gdp = gdp.loc[:, ~gdp.columns.str.contains("^Unnamed")]

# Behold kun det nødvendige
gdp = gdp[["Country Name", "Country Code"] + [col for col in gdp.columns if col.isdigit()]]

# Lav fra wide til long format
gdp = gdp.melt(
    id_vars=["Country Name", "Country Code"],
    var_name="Year",
    value_name="GDP"
)

# Omdøb kolonner
gdp = gdp.rename(columns={"Country Name": "Country"})

# Ens datatype
gdp["Year"] = pd.to_numeric(gdp["Year"], errors="coerce")

# Tjek - kan slettes, kun til kontrol
print(gdp.head())
print(gdp.columns)
print(gdp.shape)

                       Country Country Code  Year  GDP
0                        Aruba          ABW  1960  NaN
1  Africa Eastern and Southern          AFE  1960  NaN
2                  Afghanistan          AFG  1960  NaN
3   Africa Western and Central          AFW  1960  NaN
4                       Angola          AGO  1960  NaN
Index(['Country', 'Country Code', 'Year', 'GDP'], dtype='object')
(17556, 4)


In [40]:
# Clean aging
aging = aging.rename(columns={
    "Entity": "Country",
    "Code": "Country Code",
    "Age dependency ratio, old (% of working-age population)": "Age_dependency_old"
})

# Behold kun relevante kolonner
aging = aging[["Country", "Country Code", "Year", "Age_dependency_old"]]

# Ens datatype
aging["Year"] = pd.to_numeric(aging["Year"], errors="coerce")

# Tjek - kan slettes, kun til kontrol
print(aging.head())
print(aging.columns)
print(aging.shape)

       Country Country Code  Year  Age_dependency_old
0  Afghanistan          AFG  1950            5.078877
1  Afghanistan          AFG  1951            5.100585
2  Afghanistan          AFG  1952            5.114399
3  Afghanistan          AFG  1953            5.122446
4  Afghanistan          AFG  1954            5.126267
Index(['Country', 'Country Code', 'Year', 'Age_dependency_old'], dtype='object')
(19388, 4)


In [41]:
# Clean PWT
pwt = pwt.rename(columns={
    "country": "Country",
    "countrycode": "Country Code",
    "year": "Year"
})

# Behold relevante kolonner inkl. teknologi
pwt = pwt[["Country", "Country Code", "Year", "hc", "ctfp"]]

# Ens datatype
pwt["Year"] = pd.to_numeric(pwt["Year"], errors="coerce")

# Tjek - kan slettes, kun til kontrol
print(pwt.head())
print(pwt.columns)
print(pwt.shape)

  Country Country Code  Year  hc  ctfp
0   Aruba          ABW  1950 NaN   NaN
1   Aruba          ABW  1951 NaN   NaN
2   Aruba          ABW  1952 NaN   NaN
3   Aruba          ABW  1953 NaN   NaN
4   Aruba          ABW  1954 NaN   NaN
Index(['Country', 'Country Code', 'Year', 'hc', 'ctfp'], dtype='object')
(13690, 5)


In [42]:
rd_long = rd.melt(
    id_vars=["Country Name", "Country Code"],
    var_name="Year",
    value_name="rd"
)

rd_long = rd_long.rename(columns={
    "Country Name": "Country",
    "Country Code": "Country Code"
})

rd_long = rd_long[rd_long["Year"].str.isnumeric()]
rd_long["Year"] = pd.to_numeric(rd_long["Year"], errors="coerce")

rd_long = rd_long.dropna(subset=["rd"])
rd_long = rd_long[rd_long["Country Code"].str.len() == 3]
rd_long = rd_long[(rd_long["Year"] >= 1990) & (rd_long["Year"] <= 2020)]

In [43]:
rd_long[["Country Code", "Year"]].drop_duplicates().head()

,Country Code,Year
10117,ARG,1996
10121,AUS,1996
10122,AUT,1996
10123,AZE,1996
10125,BEL,1996


In [44]:
rd_sample = rd_long[["Country Code", "Year"]].drop_duplicates()

In [45]:
aging = aging.merge(rd_sample, on=["Country Code", "Year"], how="inner")
gdp   = gdp.merge(rd_sample, on=["Country Code", "Year"], how="inner")
pwt   = pwt.merge(rd_sample, on=["Country Code", "Year"], how="inner")

In [46]:
# Behold kun nødvendige kolonner
rd_long = rd_long[["Country Code", "Year", "rd"]]
gdp     = gdp[["Country Code", "Year", "GDP"]]
aging   = aging[["Country Code", "Year", "Age_dependency_old"]]
pwt     = pwt[["Country Code", "Year", "hc", "ctfp"]]

In [47]:
df = rd_long.merge(gdp, on=["Country Code", "Year"], how="inner")
df = df.merge(aging, on=["Country Code", "Year"], how="inner")
df = df.merge(pwt, on=["Country Code", "Year"], how="inner")

In [48]:
print(df.shape)
print(df.isna().sum())

(2131, 7)
Country Code            0
Year                    0
rd                      0
GDP                     4
Age_dependency_old      0
hc                    188
ctfp                  293
dtype: int64


In [49]:
df_clean = df.dropna(subset=["GDP", "hc", "rd", "ctfp"])

# kan slettes, kun til kontrol
print(df_clean.shape)
print(df_clean.isna().sum())
print(df_clean.head())

(1835, 7)
Country Code          0
Year                  0
rd                    0
GDP                   0
Age_dependency_old    0
hc                    0
ctfp                  0
dtype: int64
  Country Code  Year       rd           GDP  Age_dependency_old        hc  \
0          ARG  1996  0.41749  10495.839245           15.171934  2.627024   
1          AUS  1996  1.66218  22133.844279           17.940866  3.487049   
2          AUT  1996  1.58947  24426.764096           22.748112  3.055366   
4          BEL  1996  1.74299  22745.214587           24.452860  2.906220   
5          BFA  1996  0.15834    753.309167            6.346507  1.053331   

       ctfp  
0  0.844116  
1  0.860539  
2  0.902523  
4  1.091189  
5  0.341095  


##  Deskriptiv statistik 

In [50]:
# Deskriptive statistikker
print(df_clean.describe())

              Year            GDP  Age_dependency_old           hc  \
count  1835.000000    1835.000000         1835.000000  1835.000000   
mean   2008.647956   24841.470696           17.007143     2.880262   
std       6.962142   20844.870438            8.618107     0.536666   
min    1996.000000     580.040921            1.219402     1.053331   
25%    2003.000000    9762.766430            8.945228     2.550146   
50%    2009.000000   20235.710536           17.262000     2.968600   
75%    2015.000000   34100.267937           23.973253     3.272420   
max    2020.000000  180939.439450           48.979670     3.854527   

              ctfp  
count  1835.000000  
mean      0.737101  
std       0.264839  
min       0.148119  
25%       0.555999  
50%       0.728481  
75%       0.913206  
max       2.382832  


In [51]:
# Sortér data
df_clean = df_clean.sort_values(["Country Code", "Year"])

# Beregn årlig vækst i GDP
df_clean["gdp_growth"] = df_clean.groupby("Country Code")["GDP"].pct_change()

# Identificer recessioner
df_clean["recession"] = (df_clean["gdp_growth"] < 0).astype(int)

# Drop rækker med manglende værdier i gdp_growth
df_clean = df_clean.dropna(subset=["gdp_growth"])

In [52]:
# kan slettes, kun til kontrol
df_clean[["Country Code", "Year", "GDP", "gdp_growth", "recession", "ctfp"]].head(10)

,Country Code,Year,GDP,gdp_growth,recession,ctfp
933,ALB,2008,8469.245375,0.116624,0,0.527488
51,ARG,1997,11403.480793,0.086476,0,0.821413
119,ARG,1998,11835.676987,0.037900,0,0.777108
184,ARG,1999,11463.853308,-0.031415,1,0.726413
251,ARG,2000,11499.982800,0.003152,0,0.720260
323,ARG,2001,11117.754924,-0.033237,1,0.697821
403,ARG,2002,9953.464929,-0.104723,1,0.622472
494,ARG,2003,10933.392549,0.098451,0,0.660173
581,ARG,2004,12117.679742,0.108318,0,0.685914
669,ARG,2005,13464.812847,0.111171,0,0.714886


In [53]:
print(pwt.columns)

Index(['Country Code', 'Year', 'hc', 'ctfp'], dtype='object')


### OLS regression (Ordinary Least Squares)

In [55]:
df_clean["Age_dependency_old"] = pd.to_numeric(df_clean["Age_dependency_old"], errors="coerce")
df_clean["hc"] = pd.to_numeric(df_clean["hc"], errors="coerce")
df_clean["rd"] = pd.to_numeric(df_clean["rd"], errors="coerce")
df_clean["gdp_growth"] = pd.to_numeric(df_clean["gdp_growth"], errors="coerce")

In [56]:
import statsmodels.api as sm

X = df_clean[["Age_dependency_old", "hc", "rd"]]
X = sm.add_constant(X)

y = df_clean["gdp_growth"]

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             gdp_growth   R-squared:                       0.019
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     11.33
Date:                Fri, 10 Apr 2026   Prob (F-statistic):           2.32e-07
Time:                        12:57:40   Log-Likelihood:                 2090.8
No. Observations:                1725   AIC:                            -4174.
Df Residuals:                    1721   BIC:                            -4152.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                  0.0794      0

### FE

In [ ]:
df_clean["log_gdp"] = np.log(df_clean["GDP"])
df_panel = df_clean.set_index(["Country Code", "Year"])

fe = PanelOLS.from_formula(
    "log_gdp ~ ctfp + Age_dependency_old + hc + recession + EntityEffects + TimeEffects",
    data=df_panel
).fit()

print(fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                log_gdp   R-squared:                        0.1977
Estimator:                   PanelOLS   R-squared (Between):              0.1040
No. Observations:                3853   R-squared (Within):               0.0967
Date:                Fri, Apr 03 2026   R-squared (Overall):              0.1041
Time:                        13:39:27   Log-likelihood                    1625.1
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      227.74
Entities:                         119   P-value                           0.0000
Avg Obs:                       32.378   Distribution:                  F(4,3698)
Min Obs:                       21.000                                           
Max Obs:                       33.000   F-statistic (robust):             227.74
                            

### RE

In [ ]:
from linearmodels.panel import RandomEffects

# (df_panel har du allerede lavet)
# df_panel = df_clean.set_index(["Country Code", "Year"])

re = RandomEffects.from_formula(
    "log_gdp ~ ctfp + Age_dependency_old + hc + recession",
    data=df_panel
).fit()

print(re.summary)


                        RandomEffects Estimation Summary                        
Dep. Variable:                log_gdp   R-squared:                        0.7570
Estimator:              RandomEffects   R-squared (Between):              0.8425
No. Observations:                3853   R-squared (Within):               0.7054
Date:                Fri, Apr 03 2026   R-squared (Overall):              0.8410
Time:                        13:39:27   Log-likelihood                   -527.39
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      2997.6
Entities:                         119   P-value                           0.0000
Avg Obs:                       32.378   Distribution:                  F(4,3849)
Min Obs:                       21.000                                           
Max Obs:                       33.000   F-statistic (robust):             2997.6
                            

### Hausman test

In [ ]:
import numpy as np
from scipy.stats import chi2

# Hent koefficienter (kun fælles variable)
fe_params = fe.params[["ctfp", "Age_dependency_old", "hc", "recession"]]
re_params = re.params[["ctfp", "Age_dependency_old", "hc", "recession"]]

fe_cov = fe.cov.loc[fe_params.index, fe_params.index]
re_cov = re.cov.loc[re_params.index, re_params.index]

diff = fe_params - re_params

stat = np.dot(np.dot(diff.T, np.linalg.inv(fe_cov - re_cov)), diff)

df = len(diff)
p_value = 1 - chi2.cdf(stat, df)

print("Hausman test statistic:", stat)
print("p-value:", p_value)

Hausman test statistic: 7489.00847061802
p-value: 0.0
